# 四脚ロボットの歩行方策を Google Colab で学習する

『つくりながら学ぶ！リアルタイムOS自作入門』第13章のロボットを歩かせている
方策（PPO で学習したニューラルネット）を、**手元に GPU が無くても**学習する
ためのノートブックです。

やること:

1. 環境を作る（mjlab + MuJoCo Warp + rsl_rl）
2. **配布済みの学習済み方策を再生する** — ここまでは数分で終わります
3. 割り当てられた GPU での**学習速度を実測**し、所要時間を見積もる
4. 学習する（チェックポイントは Google Drive に置いて、切れても続けられるように）
5. 学習した方策を動画で確認し、Arduino 用の `.h` に書き出す

**先に「ランタイム → ランタイムのタイプを変更 → T4 GPU」を選んでください。**
GPU 無しでは物理エンジン（MuJoCo Warp）が動きません。

| | |
|---|---|
| 元のコード | [code/13_quadruped/rl](https://github.com/iory/build-your-own-arduino-rtos/tree/main/code/13_quadruped/rl) |
| 学習基盤 | [unitree_rl_mjlab](https://github.com/unitreerobotics/unitree_rl_mjlab)（mjlab + rsl_rl の PPO） |
| サポートページ | https://iory.github.io/build-your-own-arduino-rtos/ |


## 0. GPU の確認

ここで GPU が出てこない場合は、**ランタイム → ランタイムのタイプを変更**で
GPU を選び直してから、このセルをもう一度実行してください。


In [ ]:
import subprocess, sys

print(sys.version)
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True)
if gpu.returncode == 0:
    print("GPU:", gpu.stdout.strip())
else:
    raise SystemExit("GPU が割り当てられていません。"
                     "ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください")

## 1. コードを取ってくる

サポートページのリポジトリから第13章のコード一式（ロボットの MJCF、報酬や
環境設定の overlay、学習済み方策）を取ってきます。`--sparse` で必要な
ディレクトリだけを落とすので数秒です。


In [ ]:
%cd /content
!rm -rf /content/build-your-own-arduino-rtos
!git clone --depth 1 --filter=blob:none --sparse \
    https://github.com/iory/build-your-own-arduino-rtos.git
!git -C build-your-own-arduino-rtos sparse-checkout set code/13_quadruped
!ls build-your-own-arduino-rtos/code/13_quadruped

## 2. 場所を環境変数で教える

`rl/` は単体で動くプログラムではなく、上流の学習基盤に被せる **overlay** です。
上流のチェックアウト先へシンボリックリンクで置かれるので、機体の MJCF には
相対パスでたどり着けません。そこで場所を環境変数で渡します
（`rl/scripts/env.sh` が shell 用にやっているのと同じことです）。

**ランタイムが再起動したら、このセルからやり直してください。**
ディスクの中身（クローンとインストール）は残っていますが、環境変数は
消えています。


In [ ]:
import os

ROOT = "/content/build-your-own-arduino-rtos/code/13_quadruped"
UP = f"{ROOT}/.upstream/unitree_rl_mjlab"

os.environ["ARDUINO_QUAD_ROOT"] = ROOT
os.environ["ARDUINO_QUAD_XML"] = f"{ROOT}/arduino_os_quad_robot/mjcf/arduino_os_quad_robot.xml"
os.environ["ARDUINO_QUAD_UPSTREAM"] = UP
os.environ["MUJOCO_GL"] = "egl"          # 画面が無いので EGL で描く
os.environ["MJLAB_WARP_QUIET"] = "1"     # カーネルのコンパイルログを黙らせる

assert os.path.exists(os.environ["ARDUINO_QUAD_XML"]), "MJCF が見つかりません"
print("ROOT =", ROOT)

## 3. 学習環境を入れる（数分）

`rl/scripts/setup.sh` が上流（unitree_rl_mjlab）を clone し、`rl/` を
`src/tasks/velocity/config/arduino_quad` としてリンクし、依存を入れます。
これで `ArduinoQuad-Flat / -Walk / -Robust / -Recovery` の 4 タスクが
登録されます。

入るのは次の組み合わせです。**版は `setup.sh` が固定しています**
（上流の `setup.py` が指す mjlab 1.2.0 では、この機体のアクチュエータ設定
`viscous_damping` と指令遅れ `delay_min/max_lag` が無くて `TypeError` に
なります。mujoco を最新にさせると mujoco-warp の import が `AttributeError`
で落ち、warp-lang を最新にさせると学習開始時のカーネル生成が
`WarpCodegenKeyError` で落ちます）。

| | |
|---|---|
| mjlab | 1.3.0 |
| mujoco-warp | 3.7.0.1 |
| mujoco | 3.7.0 |
| warp-lang | 1.14.0 |
| scipy | 1.15 以上 |
| rsl-rl-lib | 5.0.1（mjlab が指定） |

`setup.sh` は上流のチェックアウトに互換パッチも 1 つ当てます。上流の
ロボット定義が mjlab 1.3.0 で消えた `update_assets` を import していて、
そのままだと `ArduinoQuad-*` の登録まで巻き添えで落ちるためです。

pip が「既存のパッケージと衝突する」という警告を出すことがありますが、
mjlab が使う torch / numpy が入っていれば問題ありません。
**「ランタイムを再起動してください」と言われたら再起動して、
セクション 2 から再実行**してください。


In [ ]:
!bash "$ARDUINO_QUAD_ROOT/rl/scripts/setup.sh" 2>&1 | tail -25

In [ ]:
# 登録されたタスクを確認する（ここで ArduinoQuad-* の 4 つが出れば環境構築は成功）
!cd "$ARDUINO_QUAD_UPSTREAM" && python -c "import src.tasks; from mjlab.tasks.registry import list_tasks; print([t for t in list_tasks() if 'Arduino' in t])"

## 4. 配布済みの学習済み方策を動かす

学習を始める前に、**環境が正しく組めているか**を確かめます。ここで動かすのは
実機（Arduino / PC 直結）に載っているのと同じネットワークで、`13_quadruped/` に
`.npz` + `.json` で入っています。再生も実機と同じ numpy 実装
（`host/quad_policy.py`）で行うので、これが歩けば
「シミュレータ・方策・エクスポート」の 3 つが揃って正常だと分かります。

初回は MuJoCo Warp が GPU カーネルをコンパイルするので、
2〜3 分かかります（2 回目からは速いです）。


In [ ]:
!cd "$ARDUINO_QUAD_UPSTREAM" && python "$ARDUINO_QUAD_ROOT/rl/scripts/record_video.py" \
    --bundle "$ARDUINO_QUAD_ROOT" \
    --vx 0.09 --steps 300 --out /content/walk_pretrained.mp4

In [ ]:
import mediapy as media

media.show_video(media.read_video("/content/walk_pretrained.mp4"), fps=50)

## 5. この GPU での学習速度を測る

Colab が割り当てる GPU は日によって（T4 / L4 / A100）変わり、学習にかかる
時間も変わります。**30 iteration だけ回して実測**し、そこから所要時間を
見積もります。最初の数 iteration はカーネルのコンパイルを含むので、
見積もりからは外します。

この 30 iteration ぶんのログは捨てても構いません（次のセクションで
本番の学習を最初から始めます）。

比較用に、RTX 4090（2048 環境）での実測は **0.77 秒/iteration、
63,700 steps/s** です。


In [ ]:
!cd "$ARDUINO_QUAD_UPSTREAM" && python scripts/train.py ArduinoQuad-Walk \
    --env.scene.num-envs 2048 \
    --agent.max-iterations 30 \
    --agent.logger tensorboard \
    --agent.run-name colab_bench 2>&1 | tee /content/bench.log

In [ ]:
import re

text = open("/content/bench.log", encoding="utf-8", errors="replace").read()
collect = [float(v) for v in re.findall(r"Collection time:\s+([\d.]+)s", text)]
learn = [float(v) for v in re.findall(r"Learning time:\s+([\d.]+)s", text)]
fps = [int(v) for v in re.findall(r"Steps per second:\s+(\d+)", text)]
assert collect and len(collect) == len(learn), "ログを読めませんでした（上のセルの出力を確認してください）"

# 最初の 10 iteration はカーネルのコンパイルを含むので捨てる
warm = 10 if len(collect) > 15 else 0
per_iter = sum(c + l for c, l in zip(collect[warm:], learn[warm:])) / len(collect[warm:])

print(f"1 iteration あたり {per_iter:.2f} 秒（{len(collect) - warm} iteration の平均）")
print(f"シミュレーション速度 {sum(fps[warm:]) / len(fps[warm:]):,.0f} steps/s\n")
print("  iteration     見積もり時間")
for it in (500, 1000, 1500, 3000, 4500):
    h = per_iter * it / 3600
    print(f"  {it:>9,}   {h:>6.1f} 時間")
print("\n書籍の方策は 4498 iteration（4096 環境）まで回したものです。")
print("Colab は無料枠だと数時間で切れるので、次のセクションで")
print("チェックポイントを Google Drive に置いて、続きから再開できるようにします。")

## 6. チェックポイントを Google Drive に置く

Colab のセッションは切れます（無料枠では数時間、放置すると 90 分程度）。
切れるとローカルディスクは消えるので、**学習ログとチェックポイントを Drive
に逃がして**おきます。ここで作ったリンクのおかげで、次のセクションの学習は
そのまま Drive に書かれます。

Drive を使いたくない場合はこのセルを飛ばせます。その場合、セッションが
切れた時点で学習結果は失われます。


In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")

os.environ["LOGS"] = "/content/drive/MyDrive/arduino_quad_rl/logs"
!mkdir -p "$LOGS"
!rm -rf "$ARDUINO_QUAD_UPSTREAM/logs"
!ln -sfn "$LOGS" "$ARDUINO_QUAD_UPSTREAM/logs"
!ls -l "$ARDUINO_QUAD_UPSTREAM/logs/"

## 7. 学習する

```
python scripts/train.py ArduinoQuad-Walk --env.scene.num-envs 2048 ...
```

- `ArduinoQuad-Walk` が本命（平地 + 歩容シェーピング）です。タスクは他に
  `-Flat`（素の velocity レシピ）、`-Robust`（実機前の頑健化 fine-tune）、
  `-Recovery`（転倒からの起き上がり）があります
- `--env.scene.num-envs` は並列に回す機体の数。書籍は 4096 ですが、
  T4 では 2048 のほうが 1 iteration が短く、様子を見ながら進めやすいです
  （メモリに余裕があれば増やして、5 章の実測をやり直してください）
- `--agent.logger tensorboard` は必須です。**既定は wandb** で、
  Colab では認証待ちで止まります
- `--agent.save-interval 50` で 50 iteration ごとに Drive へ保存します

下のセルは実行するとその iteration 数ぶん動き続けます。**ブラウザのタブを
閉じないでください**（閉じるとセッションが切れます）。途中で止めたく
なったら停止ボタンを押してかまいません。保存済みのチェックポイントから
セクション 8 で再開できます。

**1500 iteration で、指令した速度で歩く方策になります。** 実際に回して、
100 iteration ごとのチェックポイントを指令 0.09 m/s で 6 秒ずつ再生した
実測が次のとおりです（RTX 4090、2048 環境、1 シード）。

| iteration | 実測 前進 [m/s] | iteration | 実測 前進 [m/s] |
|---|---|---|---|
| 0 | -0.000 | 800 | 0.086 |
| 100 | **-0.219** | 900 | 0.071 |
| 200 | -0.194 | 1000 | 0.072 |
| 300 | 0.005 | 1100 | 0.096 |
| 400 | 0.014 | 1200 | 0.072 |
| 500 | 0.046 | 1300 | 0.086 |
| 600 | 0.046 | 1400 | 0.101 |
| 700 | 0.058 | 1499 | **0.089** |

**100〜200 iteration では後ろに進みます**（前進報酬より先に「転ばない」を
覚えるため）。前進に転じるのが 300 付近、指令の速度に届くのが 800 前後です。
各点は 1 エピソードの測定なので、隣り合う点の ±0.02 m/s は誤差です。


In [ ]:
!cd "$ARDUINO_QUAD_UPSTREAM" && python scripts/train.py ArduinoQuad-Walk \
    --env.scene.num-envs 2048 \
    --agent.max-iterations 1500 \
    --agent.save-interval 50 \
    --agent.logger tensorboard \
    --agent.run-name colab

### 学習曲線を見る

`Train/mean_reward` が上がっているか、`Perf/total_fps` が落ちていないかを
見ます。学習中に別のセルとして実行してもかまいません。


In [ ]:
LOGDIR = f"{UP}/logs/rsl_rl/arduino_quad_velocity"

%load_ext tensorboard
%tensorboard --logdir $LOGDIR

## 8. 切れた続きから再開する

セッションが切れたら、**セクション 0 → 1 → 2 → 3 → 6 を実行し直して**から
（Drive のリンクを張り直すところまで）、このセルを実行します。

- `--agent.resume True` — mjlab は真偽値をフラグではなく値で受け取ります
  （`--agent.no-resume` のような書き方はできません）
- `--agent.load-run` は Drive に残っている実行ディレクトリ名です。
  正規表現として扱われ、複数一致したら名前順で最後のものが選ばれます
- `--agent.load-checkpoint` を省くと、その run の最後の `model_*.pt` が読まれます


In [ ]:
!ls -1 "$ARDUINO_QUAD_UPSTREAM/logs/rsl_rl/arduino_quad_velocity/"

In [ ]:
RUN = ""   # 例: "2026-08-31_10-00-00_colab"（空のままだと最新の run を継ぐ）
os.environ["RUN"] = RUN or ".*"

!cd "$ARDUINO_QUAD_UPSTREAM" && python scripts/train.py ArduinoQuad-Walk \
    --env.scene.num-envs 2048 \
    --agent.max-iterations 1500 \
    --agent.save-interval 50 \
    --agent.logger tensorboard \
    --agent.resume True \
    --agent.load-run "$RUN"

## 9. 学習した方策を動画で見る

`play.py` はビューアを開いてブロックするので Colab では使えません。
代わりに `rl/scripts/record_video.py` でオフスクリーン再生して mp4 にします。

`--vx` は前進指令 [m/s]、`--wz` は旋回指令 [rad/s] です。この機体の学習時の
指令上限は前進 0.09 m/s、旋回 0.4 rad/s なので、その範囲で試してください
（**学習していない速度を指令すると、下手に歩くのではなく壊れた動きになります**）。


In [ ]:
import glob, os

ckpts = sorted(glob.glob(f"{UP}/logs/rsl_rl/arduino_quad_velocity/*/model_*.pt"),
               key=lambda p: (os.path.dirname(p), int(p.split("model_")[-1][:-3])))
assert ckpts, "チェックポイントがありません（セクション 7 を先に）"
CKPT = ckpts[-1]

# シェルへは環境変数で渡す。`!` の継続行（\ で折り返した行）の中では
# {CKPT} 形式の置換が効かず、文字列のまま渡ってしまう。
os.environ["CKPT"] = CKPT

print("使うチェックポイント:", CKPT)

In [ ]:
!cd "$ARDUINO_QUAD_UPSTREAM" && python "$ARDUINO_QUAD_ROOT/rl/scripts/record_video.py" \
    --ckpt "$CKPT" --vx 0.09 --steps 300 --out /content/walk_mine.mp4

In [ ]:
import mediapy as media

media.show_video(media.read_video("/content/walk_mine.mp4"), fps=50)

## 10. Arduino 用の `.h` に書き出す

実機に持っていく形にします。`export_quad_policy.py` は、重みだけでなく
**観測の並び・履歴の順序・正規化の統計・home 角・action スケール**まで、
学習時の環境から読み出してヘッダに書き込みます。ここを人間が書き写すと、
エラーも出ないまま「震えるだけのロボット」ができあがるためです。
書き出したあと、numpy で再実装した順伝播と PyTorch の出力を突き合わせて
検証します。

出てくるもの:

| ファイル | 中身 |
|---|---|
| `arduino_quad_policy.h` | 重み + 前向き計算 + レイアウト定数（`13_quadruped/include/` に置き換える） |
| `arduino_quad_policy.json` | 同じ情報の機械可読版 |
| `arduino_quad_policy.npz` | PC 直結版（`host/quad_host.py`）が読む重み |

使い方は `13_quadruped/README.md` を読んでください。**最初は必ず機体を吊るして**
から動かすこと。


In [ ]:
!cd "$ARDUINO_QUAD_UPSTREAM" && python "$ARDUINO_QUAD_ROOT/host/export_quad_policy.py" \
    "$CKPT" /content/exported ArduinoQuad-Walk
!ls -l /content/exported

In [ ]:
from google.colab import files

files.download("/content/exported/arduino_quad_policy.h")
files.download("/content/exported/arduino_quad_policy.npz")
files.download("/content/exported/arduino_quad_policy.json")

## つまずいたら

| 症状 | 原因と対処 |
|---|---|
| `GPU が割り当てられていません` | ランタイム → ランタイムのタイプを変更 → T4 GPU |
| wandb のログインを求められる | `--agent.logger tensorboard` が抜けています |
| `MJCF が見つかりません` | セクション 2 の環境変数セルを実行し直す（再起動で消えます） |
| `ModuleNotFoundError: mjlab` | セクション 3 のインストールが途中で切れています。もう一度 |
| `TypeError: ... viscous_damping` / `AttributeError: ... mjENBL_MULTICCD` | mjlab / mujoco の版がずれています。`setup.sh` を通さずに pip を叩くとこうなります。セクション 3 の表の版に揃えてください |
| 動画が真っ黒 | `MUJOCO_GL=egl` が設定されていません（セクション 2） |
| セッションが切れた | セクション 0→1→2→3→6 をやり直して、セクション 8 で再開 |
| `home 姿勢が環境と方策で違います` | 方策と環境の学習設定が食い違っています。メッセージが出す `ARDUINO_QUAD_HOME_HEIGHT=...` を付けて再実行 |

質問は
[GitHub Issues](https://github.com/iory/build-your-own-arduino-rtos/issues)
へどうぞ。
